# Лабораторная работа № 4
## Разработка интерактивного дашборда

[Дашборд в DataLens](https://datalens.yandex/s7b0sx1agj2yb?_share_link=public)

В дашборде представлены исходные данные, результаты трёх методов кластеризации и сравнение их качества.

Для построения графиков подготовлены CSV-файлы. Расчёты кластеризации выполнены в предыдущих лабораторных работах.

## Подготовка данных

Используем результаты предыдущих лабораторных работ.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('Леденцы.dat', sep=';')
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11
0,4,5,5,5,3,3,3,3,3,3,3
1,5,4,5,5,3,3,3,3,3,3,3
2,5,5,4,5,3,3,3,3,3,3,3
3,5,4,5,5,3,3,3,3,3,3,3
4,4,5,5,5,3,3,3,3,3,3,3


## Результаты кластеризации

Объединим результаты трёх методов в одну таблицу для DataLens.

In [4]:
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

ward = fcluster(linkage(df, method='ward'), 4, criterion='maxclust')
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(df)
db = DBSCAN(eps=1.8, min_samples=12).fit_predict(df)

pca = PCA(n_components=2).fit_transform(df)

objects = df.copy()
objects['PCA1'] = pca[:, 0]
objects['PCA2'] = pca[:, 1]
objects['Ward'] = ward
objects['KMeans'] = km
objects['DBSCAN'] = db
objects.insert(0, 'ID', range(1, len(df) + 1))

objects.to_csv('ЛР4_объекты.csv', index=False, encoding='utf-8-sig')

results = objects.melt(
    id_vars=['ID', 'PCA1', 'PCA2'],
    value_vars=['Ward', 'KMeans', 'DBSCAN'],
    var_name='Метод',
    value_name='Кластер'
)

results['Метод'] = results['Метод'].replace({
    'Ward': 'Иерархическая',
    'KMeans': 'K-means'
})

results.to_csv('ЛР4_результаты.csv', index=False, encoding='utf-8-sig')

results.head()

,ID,PCA1,PCA2,Метод,Кластер
0,1,2.451419,-1.227245,Иерархическая,2
1,2,2.445264,-1.198855,Иерархическая,2
2,3,2.426339,-1.199295,Иерархическая,2
3,4,2.445264,-1.198855,Иерархическая,2
4,5,2.451419,-1.227245,Иерархическая,2


## Данные для графиков

Подготовим распределения признаков и средние оценки по кластерам.

In [5]:
cols = list(df.columns)

distributions = objects[['ID'] + cols].melt(
    id_vars='ID',
    var_name='Признак',
    value_name='Оценка'
)

distributions.to_csv('ЛР4_распределения.csv', index=False, encoding='utf-8-sig')

profiles = pd.concat([
    objects.groupby(method)[cols].mean()
    .reset_index()
    .rename(columns={method: 'Кластер'})
    .assign(Метод=method)
    for method in ['Ward', 'KMeans', 'DBSCAN']
], ignore_index=True)

profiles.to_csv('ЛР4_профили.csv', index=False, encoding='utf-8-sig')

profiles.head()

,Кластер,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,Метод
0,1,2.824074,2.833333,2.768519,2.620370,2.611111,2.583333,2.750000,2.916667,4.953704,4.814815,4.861111,Ward
1,2,4.845455,4.800000,4.836364,4.681818,3.036364,3.036364,3.000000,2.990909,3.072727,3.045455,3.045455,Ward
2,3,3.000000,2.966102,2.966102,2.915254,2.915254,2.949153,4.915254,4.915254,3.000000,3.000000,3.000000,Ward
3,4,3.000000,3.000000,3.000000,3.000000,4.852459,4.852459,3.000000,2.918033,2.754098,2.754098,2.754098,Ward
4,0,2.824074,2.833333,2.768519,2.620370,2.611111,2.583333,2.750000,2.916667,4.953704,4.814815,4.861111,KMeans


## Сравнение методов

Рассчитаем показатели качества кластеризации.

In [6]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

metrics = []

for method, labels in [('Ward', ward), ('KMeans', km), ('DBSCAN', db)]:
    mask = labels != -1

    metrics.append({
        'Метод': method,
        'Кластеры': len(set(labels) - {-1}),
        'Silhouette': silhouette_score(df[mask], labels[mask]),
        'Davies_Bouldin': davies_bouldin_score(df[mask], labels[mask]),
        'Calinski_Harabasz': calinski_harabasz_score(df[mask], labels[mask]),
        'Шум': sum(~mask),
        'Доля_шума': round(sum(~mask) / len(df) * 100, 2)
    })

metrics = pd.DataFrame(metrics)
metrics.to_csv('ЛР4_метрики.csv', index=False, encoding='utf-8-sig')

metrics.round(3)

,Метод,Кластеры,Silhouette,Davies_Bouldin,Calinski_Harabasz,Шум,Доля_шума
0,Ward,4,0.727,0.433,720.103,0,0.00
1,KMeans,4,0.727,0.433,720.103,0,0.00
2,DBSCAN,4,0.745,0.408,789.861,10,2.96


## Подбор параметров

Подготовим данные для графиков K-means и DBSCAN.

In [7]:
from sklearn.metrics import silhouette_score

km_scores = []

for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(df)
    km_scores.append([k, silhouette_score(df, labels)])

km_data = pd.DataFrame(km_scores, columns=['k', 'Silhouette'])
km_data.to_csv('ЛР4_подбор_KMeans.csv', index=False, encoding='utf-8-sig')

db_scores = []

for eps in [1, 1.4, 1.8, 2, 2.2, 2.5]:
    for m in [5, 8, 12, 20]:
        labels = DBSCAN(eps=eps, min_samples=m).fit_predict(df)
        mask = labels != -1
        n = len(set(labels) - {-1})

        score = silhouette_score(df[mask], labels[mask]) if 2 <= n < sum(mask) else np.nan
        db_scores.append([eps, m, n, sum(~mask), score])

db_data = pd.DataFrame(
    db_scores,
    columns=['eps', 'min_samples', 'Кластеры', 'Шум', 'Силуэт']
)

db_data.to_csv('ЛР4_подбор_eps.csv', index=False, encoding='utf-8-sig')

db_data.head()

,eps,min_samples,Кластеры,Шум,Силуэт
0,1.0,5,9,44,0.898203
1,1.0,8,8,50,0.896082
2,1.0,12,6,69,0.904873
3,1.0,20,4,100,0.911342
4,1.4,5,9,44,0.898203


## Результат

Подготовлены CSV-файлы для DataLens.

Дашборд содержит исходные данные, результаты кластеризации, сравнение методов и выводы.